In [4]:
import os
import pandas as pd
import requests
from time import sleep

RAW_DIR = "../data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

TRAIN_YEARS = list(range(2015, 2025))   
TEST_YEARS = [2025]




In [5]:
def fetch_year(year):
    print(f"\n Fetching {year}")

    offset = 0
    limit = 50000
    all_data = []

    while True:
        url = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"

        headers = {"User-Agent": "IT5006-Team9-Project"}
        

        params = {
            "$limit": limit,
            "$offset": offset,
            "$where": f"year={year}",
            "$select": "id,date,primary_type,latitude,longitude"
        }

        r = requests.get(url, headers=headers, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()

        if not data:
            break

        all_data.extend(data)
        offset += limit

        print(f"  downloaded {offset} rows")

        sleep(0.2)  

    df = pd.DataFrame(all_data)
    print(f"{year} done: {df.shape}")

    return df


In [7]:
train_file = os.path.join(RAW_DIR, "chicago_2015_2024.parquet")

if os.path.exists(train_file):
    print("Train data already exists")
else:
    dfs = []
    for y in TRAIN_YEARS:
        dfs.append(fetch_year(y))

    train_df = pd.concat(dfs, ignore_index=True)

    train_df["date"] = pd.to_datetime(train_df["date"], errors="coerce")
    train_df["latitude"] = pd.to_numeric(train_df["latitude"], errors="coerce")
    train_df["longitude"] = pd.to_numeric(train_df["longitude"], errors="coerce")

    train_df = train_df.dropna(subset=["date", "latitude", "longitude"])

    train_df.to_parquet(train_file, index=False)

    print("Train data saved")



 Fetching 2015
  downloaded 50000 rows
  downloaded 100000 rows
  downloaded 150000 rows
  downloaded 200000 rows
  downloaded 250000 rows
  downloaded 300000 rows
2015 done: (264887, 5)

 Fetching 2016
  downloaded 50000 rows
  downloaded 100000 rows
  downloaded 150000 rows
  downloaded 200000 rows
  downloaded 250000 rows
  downloaded 300000 rows
2016 done: (269957, 5)

 Fetching 2017
  downloaded 50000 rows
  downloaded 100000 rows
  downloaded 150000 rows
  downloaded 200000 rows
  downloaded 250000 rows
  downloaded 300000 rows
2017 done: (269284, 5)

 Fetching 2018
  downloaded 50000 rows
  downloaded 100000 rows
  downloaded 150000 rows
  downloaded 200000 rows
  downloaded 250000 rows
  downloaded 300000 rows
2018 done: (269146, 5)

 Fetching 2019
  downloaded 50000 rows
  downloaded 100000 rows
  downloaded 150000 rows
  downloaded 200000 rows
  downloaded 250000 rows
  downloaded 300000 rows
2019 done: (261700, 5)

 Fetching 2020
  downloaded 50000 rows
  downloaded 100000 

In [8]:
test_file = os.path.join(RAW_DIR, "chicago_2025.parquet")

if os.path.exists(test_file):
    print("Test data already exists")
else:
    dfs = []
    for y in TEST_YEARS:
        dfs.append(fetch_year(y))

    test_df = pd.concat(dfs, ignore_index=True)

    test_df["date"] = pd.to_datetime(test_df["date"], errors="coerce")
    test_df["latitude"] = pd.to_numeric(test_df["latitude"], errors="coerce")
    test_df["longitude"] = pd.to_numeric(test_df["longitude"], errors="coerce")

    test_df = test_df.dropna(subset=["date", "latitude", "longitude"])

    test_df.to_parquet(test_file, index=False)

    print("Test data saved")



 Fetching 2025
  downloaded 50000 rows
  downloaded 100000 rows
  downloaded 150000 rows
  downloaded 200000 rows
  downloaded 250000 rows
2025 done: (236473, 5)
Test data saved


In [9]:
train_df = pd.read_parquet(train_file)
test_df = pd.read_parquet(test_file)

train_df.shape, test_df.shape


((2477274, 5), (236404, 5))